# Chaos Test — AVD Session Host

**Ziel:** Den AVD Session Host gezielt unter CPU-Last setzen (statt herunterzufahren) via Azure Chaos Studio.

**Aufteilung:**
- Chaos Studio (Agent-Target + Capability + Chaos-Agent-Extension + Experiment + Rollenzuweisung) wird komplett **in Bicep** definiert und über die GitHub Action **Deploy Chaos Studio** (`infra/chaos/main.bicep`) deployed.
- Das Experiment wird **hier im Notebook mit einem einzigen Start-Call** ausgelöst, überwacht und gestoppt.

**Fault:** `urn:csci:microsoft:agent:cpuPressure/1.0` — belastet die CPU von `vm-avd-cptdazavdvwan` auf ~95 % für die Experiment-Dauer (`PT10M`). Agent-basierter Fault, benötigt den Chaos-Agent (VM-Extension) + User-Assigned Identity an der VM (beides erledigt die Action).


## Variablen

In [1]:
export PREFIX=cptdazavdvwan
export RG=rg-${PREFIX}
export VM=vm-avd-${PREFIX}
export HP=hp-${PREFIX}
export EXPERIMENT=exp-cpu-${PREFIX}
export APIV=2024-01-01
export SUB=$(az account show --query id -o tsv)
echo "RG=$RG SUB=$SUB VM=$VM HP=$HP EXPERIMENT=$EXPERIMENT"


RG=rg-cptdazavdvwan SUB=ff0bb075-6c44-44ee-bb64-d46ce828c62f VM=vm-avd-cptdazavdvwan HP=hp-cptdazavdvwan EXPERIMENT=exp-cpu-cptdazavdvwan


## 0. Voraussetzung: Chaos Studio deployed

Das Experiment muss zuvor über die GitHub Action **Deploy Chaos Studio** (`infra/chaos/main.bicep`) angelegt worden sein. Diese Zelle prüft, ob das Experiment existiert.

In [2]:
az rest --method get \
  --url "https://management.azure.com/subscriptions/${SUB}/resourceGroups/${RG}/providers/Microsoft.Chaos/experiments/${EXPERIMENT}?api-version=${APIV}" \
  --query "{name:name, provisioningState:properties.provisioningState, principalId:identity.principalId}" -o json


{
  "name": "exp-cpu-cptdazavdvwan",
  "principalId": "651e530d-16de-4828-af55-dc9c68a99c49",
  "provisioningState": "Succeeded"
}


## 1. Baseline — CPU-Auslastung (vorher)

Liest die aktuelle CPU-Auslastung direkt im Gast aus (via Run Command). Erwartung: niedrige Last (Leerlauf).


In [4]:
echo "=== CPU-Auslastung im Gast (Baseline, Mittelwert über 5s) ==="
az vm run-command invoke -g $RG -n $VM \
  --command-id RunPowerShellScript \
  --scripts "\$avg=(Get-Counter '\\Processor(_Total)\\% Processor Time' -SampleInterval 1 -MaxSamples 5).CounterSamples.CookedValue | Measure-Object -Average; Write-Output ('CPU avg (5s): {0:N1} %' -f \$avg.Average); Write-Output '--- Top 5 Prozesse (CPU) ---'; Get-Process | Sort-Object CPU -Descending | Select-Object -First 5 Name,CPU | Format-Table -AutoSize | Out-String" \
  --query "value[0].message" -o tsv


=== CPU-Auslastung im Gast (Baseline, Mittelwert über 5s) ===


CPU avg (5s): 34.4 %
--- Top 5 Prozesse (CPU) ---

Name                CPU
----                ---
MsMpEng      642.359375
MsSense      389.765625
System       245.890625
MonAgentCore 145.203125
svchost       107.46875





## 1b. Plattform-Metrik — `Percentage CPU` (Host-Sicht)

Gegenstück zur In-Guest-Messung oben. Die Zelle darüber liest mit `Get-Counter` einen **5-Sekunden-Momentwert** im Gast — der kann durch kurze Spikes (z. B. Defender `MsMpEng`/`MsSense`) hoch wirken.

Diese Zelle liest die **Plattform-Metrik `Percentage CPU`** (vom Hypervisor gemessen) über `az monitor metrics list`. Wichtig für ein faires Bild:

- **Aggregation `Maximum`** zeigt Spitzen, **`Average`** den geglätteten Verlauf.
- **`--interval PT1M`** (1-Minuten-Granularität) statt der groben 24h-Ansicht im Portal.
- Zeitfenster = letzte 15 Minuten.

So entspricht die Ausgabe dem, was du im Metric Explorer siehst, wenn du dort Aggregation = Max und Granularität = 1 Min einstellst. Während des Experiments (`pressureLevel 95`, `PT10M`) liegt die Last **dauerhaft** ~95 % — anders als ein kurzer Defender-Spike.


In [5]:
VM_ID="/subscriptions/${SUB}/resourceGroups/${RG}/providers/Microsoft.Compute/virtualMachines/${VM}"
START=$(date -u -d '15 minutes ago' '+%Y-%m-%dT%H:%M:%SZ')
END=$(date -u '+%Y-%m-%dT%H:%M:%SZ')
echo "=== Percentage CPU (Plattform/Host) — $VM, $START .. $END, PT1M ==="
az monitor metrics list \
  --resource "$VM_ID" \
  --metric "Percentage CPU" \
  --aggregation Average Maximum \
  --interval PT1M \
  --start-time "$START" --end-time "$END" \
  --query "value[0].timeseries[0].data[?average!=null || maximum!=null].{time:timeStamp, avg:average, max:maximum}" -o table


=== Percentage CPU (Plattform/Host) — vm-avd-cptdazavdvwan, 2026-06-15T09:36:47Z .. 2026-06-15T09:51:47Z, PT1M ===
Time                  Avg     Max
--------------------  ------  -----
2026-06-15T09:36:00Z  3.555   4.81
2026-06-15T09:37:00Z  1.84    1.96
2026-06-15T09:38:00Z  1.715   2.0
2026-06-15T09:39:00Z  13.595  25.53
2026-06-15T09:40:00Z  1.39    1.58
2026-06-15T09:41:00Z  1.495   1.55
2026-06-15T09:42:00Z  1.99    2.0
2026-06-15T09:43:00Z  6.855   11.86
2026-06-15T09:44:00Z  2.53    3.35
2026-06-15T09:45:00Z  1.395   1.46
2026-06-15T09:46:00Z  1.72    2.05
2026-06-15T09:47:00Z  2.255   2.54
2026-06-15T09:48:00Z  2.92    4.08
2026-06-15T09:49:00Z  6.03    10.64


## 2. Experiment starten — der eine Call

Ein einziger POST-Call startet das in Bicep definierte Experiment. Danach lesen wir die jüngste Execution aus.

> **Berechtigung:** Diese Start-Zelle (und das Cancel in Abschnitt 5) genügt die Rolle **Chaos Studio Operator** auf dem Experiment. Mitglieder der Gruppe `grp-avd-chaos` (z. B. `jesse`) können sie ohne weitere Rechte ausführen — der `start`-Call benötigt nur `Microsoft.Chaos/experiments/start/action`.


In [6]:
echo "=== Starte Experiment $EXPERIMENT ==="
az rest --method post \
  --url "https://management.azure.com/subscriptions/${SUB}/resourceGroups/${RG}/providers/Microsoft.Chaos/experiments/${EXPERIMENT}/start?api-version=${APIV}" \
  -o json
echo ""
echo "Warte, bis die Execution erscheint..."
sleep 10
az rest --method get \
  --url "https://management.azure.com/subscriptions/${SUB}/resourceGroups/${RG}/providers/Microsoft.Chaos/experiments/${EXPERIMENT}/executions?api-version=${APIV}" \
  --query "reverse(sort_by(value,&properties.startedAt))[0].{id:name,status:properties.status,startedAt:properties.startedAt}" -o json


=== Starte Experiment exp-cpu-cptdazavdvwan ===



Warte, bis die Execution erscheint...
{
  "id": "234F02DA-08CC-421A-83D5-490F74734C66",
  "startedAt": "2026-06-15T09:52:01.0275007+00:00",
  "status": "PreProcessing"
}


## 3. Experiment-Status überwachen

Mehrfach ausführbar. Status-Werte u.a.: `Running`, `Success`, `Failed`, `Cancelled`.

In [11]:
az rest --method get \
  --url "https://management.azure.com/subscriptions/${SUB}/resourceGroups/${RG}/providers/Microsoft.Chaos/experiments/${EXPERIMENT}/executions?api-version=${APIV}" \
  --query "reverse(sort_by(value,&properties.startedAt))[0:3].{id:name,status:properties.status,startedAt:properties.startedAt,stoppedAt:properties.stoppedAt}" -o table


Status               StartedAt
-------------------  ---------------------------------
PreProcessingQueued  2026-06-10T16:21:24.3733321+00:00


## 4. Auswirkung beobachten — CPU unter Last

Erwartung nach ~1–2 Min: CPU Total nahe dem konfigurierten `pressureLevel` (~95 %). Mehrfach ausführbar.


In [7]:
echo "=== CPU-Auslastung im Gast (während Experiment, Mittelwert über 5s) ==="
az vm run-command invoke -g $RG -n $VM \
  --command-id RunPowerShellScript \
  --scripts "\$avg=(Get-Counter '\\Processor(_Total)\\% Processor Time' -SampleInterval 1 -MaxSamples 5).CounterSamples.CookedValue | Measure-Object -Average; Write-Output ('CPU avg (5s): {0:N1} %' -f \$avg.Average); Write-Output '--- Top 5 Prozesse (CPU) ---'; Get-Process | Sort-Object CPU -Descending | Select-Object -First 5 Name,CPU | Format-Table -AutoSize | Out-String" \
  --query "value[0].message" -o tsv


=== CPU-Auslastung im Gast (während Experiment, Mittelwert über 5s) ===


CPU avg (5s): 29.7 %
--- Top 5 Prozesse (CPU) ---

Name                CPU
----                ---
MsMpEng      645.984375
MsSense      395.921875
System       249.265625
MonAgentCore 148.703125
lsass             108.5





## 5. Experiment vorzeitig stoppen (optional)

Bricht das laufende Experiment ab. Der Chaos-Agent beendet die CPU-Last, die Auslastung normalisiert sich danach von selbst.


In [ ]:
az rest --method post \
  --url "https://management.azure.com/subscriptions/${SUB}/resourceGroups/${RG}/providers/Microsoft.Chaos/experiments/${EXPERIMENT}/cancel?api-version=${APIV}" \
  -o json
echo "Cancel ausgelöst."


## 6. Recovery prüfen

Nach Ablauf der Dauer (oder nach Cancel) lässt die CPU-Last nach. Diese Zelle prüft, dass die Auslastung wieder im Leerlauf-Bereich liegt.


In [ ]:
echo "=== CPU-Auslastung im Gast (Recovery) ==="
az vm run-command invoke -g $RG -n $VM \
  --command-id RunPowerShellScript \
  --scripts "\$c=(Get-Counter '\\Processor(_Total)\\% Processor Time').CounterSamples.CookedValue; Write-Output ('CPU Total: {0:N1} %' -f \$c)" \
  --query "value[0].message" -o tsv


## 7. RBAC-Validierung — wer darf was?

Prüft die effektiven Rollenzuweisungen für die AVD- und Chaos-Ressourcen. Erwartung nach dem Deployment: Die Entra-Sicherheitsgruppen erscheinen mit genau einer Rolle pro Scope (Least Privilege):

| Scope | Rolle | Gruppe |
| --- | --- | --- |
| Application Group `dag-cptdazavdvwan` | Desktop Virtualization User | `grp-avd-users` |
| Session Host VM `vm-avd-cptdazavdvwan` | Virtual Machine User Login | `grp-avd-users` |
| Resource Group `rg-cptdazavdvwan` | Reader | `grp-avd-users` |
| Chaos Experiment `exp-cpu-cptdazavdvwan` | Chaos Studio Operator | `grp-avd-chaos` |

Mitglieder (z. B. `jesse`) werden über die Gruppenmitgliedschaft verwaltet — kein erneutes Deployment nötig.


In [ ]:
export APPGROUP=dag-${PREFIX}
# Gruppen-Object-IDs: bevorzugt aus Env (wie im Deployment), sonst per Anzeigename auflösen.
export AVD_GROUP=${AVD_USER_GROUP_OBJECT_ID:-$(az ad group show --group grp-avd-users --query id -o tsv 2>/dev/null)}
export CHAOS_GROUP=${CHAOS_OPERATOR_GROUP_OBJECT_ID:-$(az ad group show --group grp-avd-chaos --query id -o tsv 2>/dev/null)}
echo "AVD_GROUP=$AVD_GROUP  CHAOS_GROUP=$CHAOS_GROUP"

EXP_ID="/subscriptions/${SUB}/resourceGroups/${RG}/providers/Microsoft.Chaos/experiments/${EXPERIMENT}"
APP_ID="/subscriptions/${SUB}/resourceGroups/${RG}/providers/Microsoft.DesktopVirtualization/applicationGroups/${APPGROUP}"
VM_ID="/subscriptions/${SUB}/resourceGroups/${RG}/providers/Microsoft.Compute/virtualMachines/${VM}"
RG_ID="/subscriptions/${SUB}/resourceGroups/${RG}"

echo ""
echo "=== Application Group ($APPGROUP) — erwartet: Desktop Virtualization User (Group) ==="
az role assignment list --scope "$APP_ID" \
  --query "[].{principal:principalName, type:principalType, role:roleDefinitionName}" -o table

echo ""
echo "=== Session Host VM ($VM) — erwartet: Virtual Machine User Login (Group) ==="
az role assignment list --scope "$VM_ID" \
  --query "[].{principal:principalName, type:principalType, role:roleDefinitionName}" -o table

echo ""
echo "=== Resource Group ($RG) — erwartet: Reader (Group) ==="
az role assignment list --scope "$RG_ID" \
  --query "[?principalType=='Group'].{principal:principalName, type:principalType, role:roleDefinitionName}" -o table

echo ""
echo "=== Chaos Experiment ($EXPERIMENT) — erwartet: Chaos Studio Operator (Group) ==="
az role assignment list --scope "$EXP_ID" \
  --query "[].{principal:principalName, type:principalType, role:roleDefinitionName}" -o table
